# Step 2: Create Variables

Kilian Lüders & Hannah Birkenkötter

Creates main variables for the analysis and saves tables.

**Steps:**
1. Load and Filter Data
2. Create Variables
3. Group on Resolution Level (export for figures and further analysis)
4. Create Tables for the Publication

**Input:**
- `data/full_data.pkl` – textdata from `1_data_preparation.ipynb` on paragraph level (59685 rows x 3 columns); columns: `segmentClass`, `text`, `doc`
- `data/phrases.csv` – handcoding of resolution phrases

**Output:**
- `data/full_data_vars.pkl` - full data with new variables on paragraph level (55065 rows x 21 columns).
- `data/data_res.pkl` - data with new variables on resolution level (2721 rows x 16 columns).
- `figures/data_length.csv` – data for figure 1.
- `figures/data_res.csv` – data for figures 2, 3, 4, and 5.
- `tables/data_res.xlsx` – data for online appendix.
- `tables/tab1_art25.tex` `tables/tab1_art25.xlsx` - table 1 in the publication.
- `tables/tab2_decade.tex` `tables/tab2_decade.xlsx` - table 2 in the publication.
- `tables/tab3_decade.tex` `tables/tab3_decade.xlsx` - table 3 in the publication.

In [1]:
import re
import numpy as np
import pandas as pd

## 1. Load and Filter Data

In [2]:
# text data
full_data = pd.read_pickle("data/full_data.pkl")
full_data['content'] = (full_data['segmentClass'] == "content").astype(int)
full_data['content_num'] = (full_data['segmentClass'] == "content_num").astype(int)
full_data['len'] = full_data.text.apply(lambda x: len(x))
full_data.head()

,segmentClass,text,doc,content,content_num,len
0,head,MILITARY STAFF COMMITTEE 1 (1946) Resolution o...,S_RES_0001_1946,0,0,399
1,content,Therefore,S_RES_0001_1946,1,0,9
2,content_num,Requests the permanent members of the Security...,S_RES_0001_1946,0,1,169
3,content_num,Directs that the Chiefs of Staff or their repr...,S_RES_0001_1946,0,1,141
4,content_num,Directs the Military Staff Committee thereupon...,S_RES_0001_1946,0,1,224


In [3]:
# handcoding of UNSC Resolution phrases
phrases_pp = pd.read_csv("data/phrases.csv", index_col=0)
phrases_pp = phrases_pp[phrases_pp['type'] == "pp"]['phrase'].to_list()
phrases_pp = [e for e in phrases_pp if "ing" in e]

### 1.1. Check Data

In [4]:
# number of resolutions
full_data.doc.nunique()

2721

In [5]:
# check -> every file has at least one pp or op
full_data[full_data.segmentClass.isin(['content', 'content_num'])].doc.nunique()

2721

In [6]:
full_data.value_counts('segmentClass')

segmentClass
content_num    29379
content        25686
head            2721
heading         1070
tail             829
Name: count, dtype: int64

### 1.2. Resolutions Length

In [7]:
data_doc = full_data.groupby(['doc']).agg({'content': "sum",
                                           'content_num': "sum"}).reset_index()
data_doc['content_sum'] = data_doc.content + data_doc.content_num
data_doc['year'] = data_doc.doc.apply(lambda x: x[-4:])
data_doc

,doc,content,content_num,content_sum,year
0,S_RES_0001_1946,1,3,4,1946
1,S_RES_0002_1946,4,0,4,1946
2,S_RES_0003_1946,7,0,7,1946
3,S_RES_0004_1946,3,0,3,1946
4,S_RES_0005_1946,2,0,2,1946
...,...,...,...,...,...
2716,S_RES_2717_2023,23,50,73,2023
2717,S_RES_2718_2023,21,16,37,2023
2718,S_RES_2719_2023,18,17,35,2023
2719,S_RES_2720_2023,14,16,30,2023


In [8]:
## save data for fig 1)
data_doc.to_csv("figures/data_length.csv")

### 1.3. Filter for relevant text segments

In [9]:
print(full_data.shape)
full_data = full_data[full_data.segmentClass.isin(["content", "content_num"])]
print(full_data.shape)

(59685, 6)
(55065, 6)


## 2. Create Variables

In [10]:
# „Decides to remain actively …“ (bool)
full_data['decides_act'] = full_data.text.apply(lambda x: bool(re.search(r'(Decides to remain actively|Decides to remain seized of the matter)',x))).astype(int)

# anywhere: „decides“ (bool) -> without "decides_act"
full_data['decides'] = False
full_data.loc[full_data['decides_act'] == 0,'decides'] = full_data.loc[full_data['decides_act'] == 0,'text'].apply(lambda x: bool(re.search(r'[Dd]ecides',x)))
full_data['decides'] = full_data.decides.astype(int)

# anywhere: „Article 25“ (bool)
full_data['art25_charter'] = full_data.text.apply(lambda x: bool(re.search(r'Article 25.*Charter',x))).astype(int)

# anywhere: „Chapter VII“ (bool)
full_data['chpVII'] = full_data.text.apply(lambda x: bool(re.search(r'[Cc]hapter VII[\s,](?!of \w+ report)',x))).astype(int)

# anywhere: „Article 39“ (bool)
full_data['art39_charter'] = full_data.text.apply(lambda x: bool(re.search(r'Article.*39.*Charter',x))).astype(int)

# anywhere: „Article 40“ (bool)
full_data['art40_charter'] = full_data.text.apply(lambda x: bool(re.search(r'Article.*40.*Charter',x))).astype(int)

# anywhere: „Article 41“ (bool)
full_data['art41_charter'] = full_data.text.apply(lambda x: bool(re.search(r'Article.*41.*Charter',x))).astype(int)

# anywhere: „Article 42“ (bool)
full_data['art42_charter'] = full_data.text.apply(lambda x: bool(re.search(r'Article.*42.*Charter',x))).astype(int)

# anywhere: „Article 39 language“ (bool)
full_data['art39lang'] = (full_data.text.apply(lambda x: bool(re.search(r'(?:threat to peace|threat to the peace|threat to international peace|breach of the peace|breach of international peace|aggression|aggressive act|international peace and security|threat to regional peace)',x))) +
                          full_data.text.apply(lambda x: bool(re.search(r'aggress.*acts?[\s,]',x)))).astype(int)
full_data['breach_int'] = full_data.text.apply(lambda x: bool(re.search(r'(?:breach of international peace)',x))).astype(int)


full_data['int_peace_sec'] = full_data.text.apply(lambda x: bool(re.search(r'(?:international peace and security)',x))).astype(int)

full_data['art39lang_wout'] = (full_data.text.apply(lambda x: bool(re.search(r'(?:threat to peace|threat to the peace|threat to international peace|breach of the peace|breach of international peace|aggression|aggressive act|threat to regional peace)',x))) +
                          full_data.text.apply(lambda x: bool(re.search(r'aggress.*acts?[\s,]',x)))).astype(int)


def check_pp_phrases(para):
    for phrase in phrases_pp:
        if phrase in para:
            return True
    return False


# „authorizes“ (bool)
full_data['authorizes'] = full_data.text.apply(lambda x: bool(re.search(r'[Aa]uthori[sz]es',x))).astype(int)
full_data.loc[full_data['text'].apply(lambda x: check_pp_phrases(x)),'authorizes'] = 0

# requests (bool)
full_data['requests'] = full_data.text.apply(lambda x: bool(re.search(r'(?:Requests|Further requests|Also requests|also requests|, requests|and requests)',x))).astype(int)
full_data.loc[full_data['text'].apply(lambda x: check_pp_phrases(x)),'requests'] = 0

# „Demand“ (bool)
full_data['demand'] = full_data.text.apply(lambda x: bool(re.search(r'[Dd]emands',x))).astype(int)
full_data.loc[full_data['text'].apply(lambda x: check_pp_phrases(x)),'demand'] = 0

In [11]:
full_data.to_pickle("data/full_data_vars.pkl")
full_data.shape

(55065, 21)

## 3. Group on Resolution Level

Export for Figures and further Analysis

In [12]:
res_data = full_data.groupby("doc").agg({
    'decides': 'sum',
    'decides_act': 'sum',
    'art25_charter': 'sum',
    'chpVII': 'sum',
    'art39_charter': 'sum',
    'art40_charter': 'sum',
    'art41_charter': 'sum',
    'art39lang': 'sum',
    'demand': 'sum',
    'authorizes': 'sum',
    'requests': 'sum',
    'int_peace_sec': 'sum',
    'art39lang_wout': 'sum'
}).reset_index()

res_data['decides'] = res_data.decides > 0
res_data['decides_act'] = res_data.decides_act > 0
res_data['art25_charter'] = res_data.art25_charter > 0
res_data['art39_charter'] = res_data.art39_charter > 0
res_data['art40_charter'] = res_data.art40_charter > 0
res_data['art41_charter'] = res_data.art41_charter > 0
res_data['art39lang'] = res_data.art39lang > 0
res_data['chpVII'] = res_data.chpVII > 0
res_data['demand'] = res_data.demand > 0
res_data['authorizes'] = res_data.authorizes > 0
res_data['requests'] = res_data.requests > 0
res_data['int_peace_sec'] = res_data.int_peace_sec > 0
res_data['art39lang_wout'] = res_data.art39lang_wout > 0


res_data['decides'] = res_data.decides.astype(int)
res_data['decides_act'] = res_data.decides_act.astype(int)
res_data['art25_charter'] = res_data.art25_charter.astype(int)
res_data['art39_charter'] = res_data.art39_charter.astype(int)
res_data['art40_charter'] = res_data.art40_charter.astype(int)
res_data['art41_charter'] = res_data.art41_charter.astype(int)
res_data['art39lang'] = res_data.art39lang.astype(int)
res_data['chpVII'] = res_data.chpVII.astype(int)
res_data['demand'] = res_data.demand.astype(int)
res_data['authorizes'] = res_data.authorizes.astype(int)
res_data['requests'] = res_data.requests.astype(int)
res_data['int_peace_sec'] = res_data.int_peace_sec.astype(int)
res_data['art39lang_wout'] = res_data.art39lang_wout.astype(int)


res_data['year'] = res_data.doc.apply(lambda x: x[-4:]).astype(int)
res_data['decade'] = (res_data['year'] // 10) * 10
res_data

,doc,decides,decides_act,art25_charter,chpVII,art39_charter,art40_charter,art41_charter,art39lang,demand,authorizes,requests,int_peace_sec,art39lang_wout,year,decade
0,S_RES_0001_1946,0,0,0,0,0,0,0,0,0,0,1,0,0,1946,1940
1,S_RES_0002_1946,0,0,0,0,0,0,0,0,0,0,1,0,0,1946,1940
2,S_RES_0003_1946,0,0,0,0,0,0,0,0,0,0,0,0,0,1946,1940
3,S_RES_0004_1946,0,0,0,0,0,0,0,1,0,0,0,1,0,1946,1940
4,S_RES_0005_1946,0,0,0,0,0,0,0,0,0,0,0,0,0,1946,1940
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2716,S_RES_2717_2023,1,1,0,1,0,0,0,1,1,1,1,1,1,2023,2020
2717,S_RES_2718_2023,1,0,0,0,0,0,0,0,0,0,1,0,0,2023,2020
2718,S_RES_2719_2023,1,0,0,0,0,0,0,1,0,1,1,1,0,2023,2020
2719,S_RES_2720_2023,0,1,0,0,0,0,0,0,1,0,1,0,0,2023,2020


In [13]:
## key dataset with all vars and one observation per resolution
res_data.to_pickle("data/data_res.pkl")
res_data.to_csv("figures/data_res.csv")
res_data.to_excel("tables/data_res.xlsx")

In [14]:
res_data[["decides", "decides_act", "art25_charter", "chpVII", "art39_charter", "art40_charter", "art41_charter", "art39lang", "demand", "authorizes", "requests", "int_peace_sec", "art39lang_wout"]].sum()

decides           1820
decides_act       1743
art25_charter       30
chpVII             896
art39_charter        6
art40_charter        6
art41_charter       49
art39lang         1100
demand             572
authorizes         295
requests          1909
int_peace_sec     1012
art39lang_wout     853
dtype: int64

## 4. Create Tables for the Publication

Data is now grouped by *decade*

### Table 1

In [15]:
decade_data = res_data.groupby("decade").agg({
    'art25_charter': 'sum',
    'chpVII': 'sum',
    'art39_charter': 'sum',
    'art39lang': 'sum',
    'art40_charter': 'sum',
    'art41_charter': 'sum',
    'decides': 'sum',
    'demand': 'sum',
    'authorizes': 'sum',
    'requests': 'sum',
    'doc': 'count'
}).reset_index().rename(columns={'doc':'n'})

decade_data

,decade,art25_charter,chpVII,art39_charter,art39lang,art40_charter,art41_charter,decides,demand,authorizes,requests,n
0,1940,0,3,2,11,3,0,8,0,1,19,78
1,1950,0,0,0,11,0,0,18,0,5,19,54
2,1960,3,1,1,22,0,2,25,9,3,54,143
3,1970,9,11,1,41,0,4,74,29,1,88,186
4,1980,1,3,1,39,1,0,97,33,6,119,185
5,1990,3,164,1,121,1,2,423,162,73,448,638
6,2000,0,279,0,305,1,12,487,129,83,452,623
7,2010,9,329,0,404,0,25,497,152,94,521,596
8,2020,5,106,0,146,0,4,191,58,29,189,218


In [16]:
decade_data[['decade', 'art25_charter', 'n']].to_excel("tables/tab1_art25.xlsx", index=False)
decade_data[['decade', 'art25_charter', 'n']].to_latex("tables/tab1_art25.tex", index=False)

### Table 2

In [17]:
output_df = decade_data[['decade', 'art25_charter', 'art39_charter', 'art40_charter',  'art41_charter', 'chpVII', 'art39lang', 'n']].copy()

# column 5 (chpVII)
percentages = output_df.iloc[:, 5].div(output_df['n'], axis=0) * 100
col = output_df.columns[5]
output_df[col] = output_df.apply(lambda row: f"{row[col]} ({row[col]/row['n']*100:.1f}%)", axis=1)

# column 6 (art39lang)
percentages = output_df.iloc[:, 6].div(output_df['n'], axis=0) * 100
col = output_df.columns[6]
output_df[col] = output_df.apply(lambda row: f"{row[col]} ({row[col]/row['n']*100:.1f}%)", axis=1)

output_df['decade'] = ["1946-1949","1950-1959", "1960-1969", "1970-1979", "1980-1989", "1990-1999", "2000-2009", "2010-2019", "2020-1923"]

#output_df.to_csv('data/decade_table_1.csv', index=False)
output_df.to_excel('tables/tab2_decade.xlsx', index=False)
output_df.to_latex('tables/tab2_decade.tex', index=False)
output_df

,decade,art25_charter,art39_charter,art40_charter,art41_charter,chpVII,art39lang,n
0,1946-1949,0,2,3,0,3 (3.8%),11 (14.1%),78
1,1950-1959,0,0,0,0,0 (0.0%),11 (20.4%),54
2,1960-1969,3,1,0,2,1 (0.7%),22 (15.4%),143
3,1970-1979,9,1,0,4,11 (5.9%),41 (22.0%),186
4,1980-1989,1,1,1,0,3 (1.6%),39 (21.1%),185
5,1990-1999,3,1,1,2,164 (25.7%),121 (19.0%),638
6,2000-2009,0,0,1,12,279 (44.8%),305 (49.0%),623
7,2010-2019,9,0,0,25,329 (55.2%),404 (67.8%),596
8,2020-1923,5,0,0,4,106 (48.6%),146 (67.0%),218


### Table 3

In [18]:
output_df = decade_data[['decade', 'decides','demand', 'authorizes', 'requests', 'n']].copy()

percentages = output_df.iloc[:, 1:-1].div(output_df['n'], axis=0) * 100

for col in output_df.columns[1:-1]:  
    output_df[col] = output_df.apply(lambda row: f"{row[col]} ({row[col]/row['n']*100:.1f}%)", axis=1)

output_df['decade'] = ["1946-1949","1950-1959", "1960-1969", "1970-1979", "1980-1989", "1990-1999", "2000-2009", "2010-2019", "2020-2023"]

#output_df.to_csv('data/decade_table_1.csv', index=False)
output_df.to_excel('tables/tab3_decade.xlsx', index=False)
output_df.to_latex('tables/tab3_decade.tex', index=False)
output_df

,decade,decides,demand,authorizes,requests,n
0,1946-1949,8 (10.3%),0 (0.0%),1 (1.3%),19 (24.4%),78
1,1950-1959,18 (33.3%),0 (0.0%),5 (9.3%),19 (35.2%),54
2,1960-1969,25 (17.5%),9 (6.3%),3 (2.1%),54 (37.8%),143
3,1970-1979,74 (39.8%),29 (15.6%),1 (0.5%),88 (47.3%),186
4,1980-1989,97 (52.4%),33 (17.8%),6 (3.2%),119 (64.3%),185
5,1990-1999,423 (66.3%),162 (25.4%),73 (11.4%),448 (70.2%),638
6,2000-2009,487 (78.2%),129 (20.7%),83 (13.3%),452 (72.6%),623
7,2010-2019,497 (83.4%),152 (25.5%),94 (15.8%),521 (87.4%),596
8,2020-2023,191 (87.6%),58 (26.6%),29 (13.3%),189 (86.7%),218
